# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [3]:
# База. 1. Самый дорогой заказ
max_index = int(amount.argmax())
print(f"Позиция (с нуля): {max_index}, сумма: {amount[max_index]:,.2f} ₸")
print(f"Самый дорогой заказ оформил клиент {cust[max_index]}, город — {city[max_index]}.")

Позиция (с нуля): 1001, сумма: 9,999,990.00 ₸
Самый дорогой заказ оформил клиент TEST, город — Алматы.


In [4]:
# База. 2. Оплаченные и неоплаченные заказы
paid_total = amount[paid].sum()
unpaid_total = amount[~paid].sum()
print(f"Магазин получил {paid_total:,.2f} ₸, а сумма неоплаченных заказов — {unpaid_total:,.2f} ₸.")

Магазин получил 101,909,360.00 ₸, а сумма неоплаченных заказов — 7,974,000.00 ₸.


In [5]:
# База. 3. Заказы в Шымкенте
shymkent_orders = city == "Шымкент"
print(f"Шымкент: {shymkent_orders.sum()} заказа, или {shymkent_orders.mean():.2%} от всех заказов.")

Шымкент: 804 заказа, или 14.72% от всех заказов.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [6]:
# Стандарт. 4. Потери выручки из-за скидки
discount_orders = paid & (amount > 50_000)
loss_all = amount[discount_orders].sum() * 0.15
loss_without_test = amount[discount_orders & (cust != "TEST")].sum() * 0.15
print(f"Потери со всеми заказами: {loss_all:,.2f} ₸, без TEST: {loss_without_test:,.2f} ₸.")
print(f"Три крупных тестовых заказа завышают оценку потерь в {loss_all / loss_without_test:.1f} раза.")

Потери со всеми заказами: 5,702,422.50 ₸, без TEST: 1,202,427.00 ₸.
Три крупных тестовых заказа завышают оценку потерь в 4.7 раза.


In [7]:
# Стандарт. 5. Доля быстрых доставок по городам
fast_delivery = (days >= 1) & (days <= 2)
print("Доля заказов с доставкой за 1–2 дня от всех заказов города:")
for city_name in np.unique(city):
    city_orders = city == city_name
    fast_share = fast_delivery[city_orders].mean()
    print(f"{city_name} — {fast_share:.2%}")

Доля заказов с доставкой за 1–2 дня от всех заказов города:
Актобе — 62.22%
Алматы — 64.53%
Астана — 63.53%
Атырау — 64.38%
Караганда — 67.84%
Шымкент — 29.85%


In [8]:
# Стандарт. 6. Средний чек
# Для среднего чека берём оплаченные заказы. Для оценки покупок исключаем TEST.
# round(avg) округляет до целого тенге, поэтому выводим два знака после запятой.
customer_paid = paid & (cust != "TEST")
average_check = amount[customer_paid].mean()
average_with_test = amount[paid].mean()
print(f"Средний чек оплаченного заказа — {average_check:,.2f} ₸ без TEST и {average_with_test:,.2f} ₸ с TEST.")

Средний чек оплаченного заказа — 14,672.39 ₸ без TEST и 20,780.86 ₸ с TEST.


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж